# MLflow Experiment Tracking with DagsHub — Wine Quality (Red)

Regression task: predict the wine `quality` score from 11 physicochemical features.
Three models are trained and tracked with MLflow (parameters, metrics, model artifacts).
The **best-performing model is registered in the MLflow Model Registry**, which the
FastAPI service later loads for inference.

## Importing Packages

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import mlflow
import mlflow.sklearn

import dagshub

## DagsHub + MLflow Setup

In [2]:
dagshub.init(
    repo_owner="itswin01",
    repo_name="wine-quality-capstone-project",
    mlflow=True,
)

mlflow.set_experiment("wine-quality-regression")

# Name under which the best model will be registered in the MLflow Model Registry
REGISTERED_MODEL_NAME = "wine-quality-model"

Accessing as itswin01

Initialized MLflow to track repo "itswin01/wine-quality-capstone-project"

Repository itswin01/wine-quality-capstone-project initialized!

2026/08/09 10:53:58 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality-regression' does not exist. Creating a new experiment.


## Load Data

In [3]:
df = pd.read_csv("winequality-red.csv")
print("shape:", df.shape)
df.head()

shape: (1599, 12)


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [4]:
X = df.drop(columns=["quality"])
y = df["quality"]

## Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("train:", X_train.shape, " test:", X_test.shape)

train: (1279, 11)  test: (320, 11)


## Train Models & Log to MLflow

Each model runs inside its own MLflow run. Hyperparameters, regression metrics (MAE, RMSE, R²), and the fitted model artifact are all logged and pushed to DagsHub. The run id and R² of each run are collected so the best one can be registered afterwards.

In [10]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=200, max_depth=6, random_state=42
    ),
    "XGBRegressor": XGBRegressor(
        n_estimators=200, max_depth=6, random_state=42
    ),
}

results = []

for name, model in models.items():
    with mlflow.start_run(run_name=name) as run:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        rmse = mean_squared_error(y_test, preds) ** 0.5
        r2 = r2_score(y_test, preds)

        mlflow.log_params(model.get_params())
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)

        # artifact_path is 'model' so the model URI is runs:/<run_id>/model
        logged = mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")

        results.append({"name": name, "run_id": run.info.run_id, "r2": r2, "model_uri": logged.model_uri})
        print(f"{name:>22}  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")

2026/08/09 11:13:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


      LinearRegression  MAE=0.5035  RMSE=0.6245  R2=0.4032
🏃 View run LinearRegression at: https://dagshub.com/itswin01/wine-quality-capstone-project.mlflow/#/experiments/0/runs/bdfdc1f87a6b46b99392c4c064b3b68d
🧪 View experiment at: https://dagshub.com/itswin01/wine-quality-capstone-project.mlflow/#/experiments/0


2026/08/09 11:13:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


 RandomForestRegressor  MAE=0.4803  RMSE=0.5984  R2=0.4520
🏃 View run RandomForestRegressor at: https://dagshub.com/itswin01/wine-quality-capstone-project.mlflow/#/experiments/0/runs/58c76651f9884969af4ed37b4391ae81
🧪 View experiment at: https://dagshub.com/itswin01/wine-quality-capstone-project.mlflow/#/experiments/0


2026/08/09 11:13:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


          XGBRegressor  MAE=0.4100  RMSE=0.6029  R2=0.4438
🏃 View run XGBRegressor at: https://dagshub.com/itswin01/wine-quality-capstone-project.mlflow/#/experiments/0/runs/22695898283943b68d617f3c8d2838ca
🧪 View experiment at: https://dagshub.com/itswin01/wine-quality-capstone-project.mlflow/#/experiments/0


## Select the Best Model & Register It

The run with the highest R² is registered in the MLflow Model Registry under
`REGISTERED_MODEL_NAME`. This registered model is what the FastAPI service loads.

In [11]:
best = max(results, key=lambda r: r["r2"])
print(f"Best model: {best['name']}  (R2={best['r2']:.4f})")

model_uri = best["model_uri"]
registered = mlflow.register_model(model_uri=model_uri, name=REGISTERED_MODEL_NAME)

print(f"Registered '{REGISTERED_MODEL_NAME}' version {registered.version}")

Best model: RandomForestRegressor  (R2=0.4520)


Registered model 'wine-quality-model' already exists. Creating a new version of this model...
2026/08/09 11:16:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine-quality-model, version 1
Created version '1' of model 'wine-quality-model'.


Registered 'wine-quality-model' version 1


## Notes

- Compare the runs in the MLflow UI (Experiments tab on DagsHub) and confirm the registered model under the Models tab.
- Wine-quality regression with tree models realistically lands around R² ≈ 0.35–0.50; that is expected, not a bug.
- The FastAPI-Docker module pulls `models:/wine-quality-model/<version>` from this same registry.